In [3]:
import sys

print(sys.executable)

e:\revive_pay\.rpay\Scripts\python.exe


## Dataset Load

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment ready!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Environment ready!
Pandas: 3.0.5
NumPy: 2.4.6


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load all Olist datasets
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [6]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name:25} {df.shape}")

customers                 (99441, 5)
geolocation               (1000163, 5)
order_items               (112650, 7)
payments                  (103886, 5)
reviews                   (99224, 7)
orders                    (99441, 8)
products                  (32951, 9)
sellers                   (3095, 4)
category_translation      (71, 2)


In [7]:
for name, df in datasets.items():
    print("\n" + "=" * 30)
    print(name.upper())
    print("=" * 30)
    print(df.columns.tolist())


CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

GEOLOCATION
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

ORDER_ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

REVIEWS
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

PRODUCTS
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'prod

In [8]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("\n" + "=" * 30)
    print(name.upper())
    print("=" * 30)
    print(missing)


CUSTOMERS
Series([], dtype: int64)

GEOLOCATION
Series([], dtype: int64)

ORDER_ITEMS
Series([], dtype: int64)

PAYMENTS
Series([], dtype: int64)

REVIEWS
review_comment_title      87656
review_comment_message    58247
dtype: int64

ORDERS
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

PRODUCTS
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

SELLERS
Series([], dtype: int64)

CATEGORY_TRANSLATION
Series([], dtype: int64)


In [9]:
for name, df in datasets.items():
    print(f"{name:25} {df.duplicated().sum()} duplicates")

customers                 0 duplicates
geolocation               261831 duplicates
order_items               0 duplicates
payments                  0 duplicates
reviews                   0 duplicates
orders                    0 duplicates
products                  0 duplicates
sellers                   0 duplicates
category_translation      0 duplicates


In [10]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        
        if missing_count > 0:
            missing_summary.append({
                "dataset": name,
                "column": column,
                "missing_count": missing_count,
                "missing_percentage": round(
                    (missing_count / len(df)) * 100, 2
                )
            })

missing_df = pd.DataFrame(missing_summary)

missing_df.sort_values(
    "missing_percentage",
    ascending=False
).reset_index(drop=True)

,dataset,column,missing_count,missing_percentage
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
3,products,product_name_lenght,610,1.85
4,products,product_category_name,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,orders,order_delivered_carrier_date,1783,1.79
8,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [11]:
missing_df.sort_values(
    ["dataset", "missing_percentage"],
    ascending=[True, False]
)

,dataset,column,missing_count,missing_percentage
4,orders,order_delivered_customer_date,2965,2.98
3,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01
10,products,product_length_cm,2,0.01
11,products,product_height_cm,2,0.01


In [12]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [13]:
orders["order_status"].value_counts(normalize=True).mul(100).round(2)

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64

In [14]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [15]:
payments["payment_type"].value_counts(normalize=True).mul(100).round(2)

payment_type
credit_card    73.92
boleto         19.04
voucher         5.56
debit_card      1.47
not_defined     0.00
Name: proportion, dtype: float64

In [16]:
payments["payment_value"].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [17]:
order_payment_summary = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count")
    )
)

order_payment_summary.head()

,order_id,total_payment_value,payment_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1


In [18]:
order_analysis = orders.merge(
    order_payment_summary,
    on="order_id",
    how="left"
)

order_analysis.groupby("order_status")["total_payment_value"].agg(
    ["count", "sum", "mean", "median"]
).sort_values("sum", ascending=False)

,count,sum,mean,median
order_status,,,,
delivered,96477,15422461.77,159.856357,105.28
shipped,1107,177213.96,160.084878,103.39
canceled,625,143255.60,229.208960,110.32
unavailable,609,126479.51,207.683924,104.63
processing,301,69394.11,230.545216,144.56
invoiced,314,69137.99,220.184682,137.77
created,5,688.10,137.620000,137.60
approved,2,241.08,120.540000,120.54


In [19]:
payment_type_summary = (
    payments
    .groupby("payment_type")
    .agg(
        payment_count=("payment_value", "count"),
        total_payment_value=("payment_value", "sum"),
        avg_payment_value=("payment_value", "mean")
    )
    .sort_values("total_payment_value", ascending=False)
)

payment_type_summary

,payment_count,total_payment_value,avg_payment_value
payment_type,,,
credit_card,76795,12542084.19,163.319021
boleto,19784,2869361.27,145.034435
voucher,5775,379436.87,65.703354
debit_card,1529,217989.79,142.570170
not_defined,3,0.00,0.000000


In [20]:
payment_status = payments.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

payment_status.groupby(
    ["order_status", "payment_type"]
)["payment_value"].agg(
    ["count", "sum", "mean"]
).round(2)

count          sum    mean
order_status payment_type                            
approved     credit_card       2       241.08  120.54
canceled     boleto           95     17504.10  184.25
             credit_card     444     97375.31  219.31
             debit_card        7      2711.27  387.32
             not_defined       3         0.00    0.00
             voucher         115     25664.92  223.17
created      boleto            2       175.44   87.72
             credit_card       3       512.66  170.89
delivered    boleto        19191   2769932.58  144.33
             credit_card   74586  12101094.88  162.24
             debit_card     1486    208421.12  140.26
             voucher        5493    343013.19   62.45
invoiced     boleto           67     15330.82  228.82
             credit_card     239     51094.13  213.78
             debit_card        6       986.04  164.34
             voucher          13      1727.00  132.85
processing   boleto           70     17135.76  244.80
             credit_card     224     50904.01  227.25
             debit_card        2       349.79  174.90
             voucher          23      1004.55   43.68
shipped      boleto          209     24227.70  115.92
             credit_card     851    146195.80  171.79
             debit_card       22      2660.80  120.95
             voucher          84      4129.66   49.16
unavailable  boleto          150     25054.87  167.03
             credit_card     446     94666.32  212.26
             debit_card        6      2860.77  476.80
             voucher          47      3897.55   82.93

In [21]:
canceled_orders = order_analysis[
    order_analysis["order_status"] == "canceled"
].copy()

canceled_orders[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_approved_at",
        "total_payment_value"
    ]
].head(10)

,order_id,customer_id,order_purchase_timestamp,order_approved_at,total_payment_value
397,1b9ecfe83cdc259250e1a8aca174f0ad,6d6b50b66d79f80827b6d96751528d30,2018-08-04 14:29:27,2018-08-07 04:10:26,33.34
613,714fb133a6730ab81fa1d3c1b2007291,e3fe72696c4713d64d3c10afe71e75ed,2018-01-26 21:34:08,2018-01-26 21:58:39,96.01
1058,3a129877493c8189c59c60eb71d97c29,0913cdce793684e52bbfac69d87e91fd,2018-01-25 13:34:24,2018-01-25 13:50:20,51.00
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,2018-08-28 15:26:39,NaN,0.00
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,2018-09-20 13:54:16,NaN,191.46
1811,0966b61e30c4a07edbd7523f59b3f3e4,2fcc597b8934d99715dbfff7909dd27f,2018-05-22 18:50:55,2018-05-22 19:17:15,274.11
1819,9021cf1919f615a121410790dcce848f,7acf55df0298e1d2c31200fb4f6fb93b,2018-07-04 16:05:56,2018-07-06 02:55:16,233.87
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,2017-03-04 12:14:30,NaN,139.96
1971,a39d3db795a5cf4c8b6c9dd050f0d326,ec66df2cb66dfda07c03050470e21f69,2017-03-13 16:12:24,2017-03-13 16:12:24,146.01
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,2018-08-29 16:27:49,NaN,66.44


In [22]:
canceled_orders["total_payment_value"].describe()

count     625.000000
mean      229.208960
std       413.628019
min         0.000000
25%        65.000000
50%       110.320000
75%       215.060000
max      4809.440000
Name: total_payment_value, dtype: float64

## Building the Revive Pay Analytical Dataset

**We combine order, payment, customer, and order-item information
into a single order-level dataset for revenue recovery analysis.**

In [23]:
# ==========================================
# REVIVE PAY - ORDER LEVEL DATASET
# ==========================================

# 1. Payment features per order
payment_features = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_installments=("payment_installments", "max"),
        primary_payment_type=("payment_type", "first")
    )
    .reset_index()
)

# 2. Item features per order
item_features = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        total_item_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

# 3. Customer information
customer_features = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
].copy()

# 4. Combine everything at order level
revive_pay_df = (
    orders
    .merge(payment_features, on="order_id", how="left")
    .merge(item_features, on="order_id", how="left")
    .merge(customer_features, on="customer_id", how="left")
)

print("Revive Pay dataset created.")
print("Shape:", revive_pay_df.shape)

Revive Pay dataset created.
Shape: (99441, 20)


In [24]:
revive_pay_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,total_payment_value,payment_count,payment_installments,primary_payment_type,item_count,total_item_price,total_freight_value,unique_products,unique_sellers,customer_unique_id,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,38.71,3.0,1.0,credit_card,1.0,29.99,8.72,1.0,1.0,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,141.46,1.0,1.0,boleto,1.0,118.70,22.76,1.0,1.0,af07308b275d755c9edb36a90c618231,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,179.12,1.0,3.0,credit_card,1.0,159.90,19.22,1.0,1.0,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,72.20,1.0,1.0,credit_card,1.0,45.00,27.20,1.0,1.0,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,28.62,1.0,1.0,credit_card,1.0,19.90,8.72,1.0,1.0,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP


In [25]:
print("Rows:", len(revive_pay_df))
print("Columns:", len(revive_pay_df.columns))
print("Unique orders:", revive_pay_df["order_id"].nunique())

Rows: 99441
Columns: 20
Unique orders: 99441


In [26]:
revive_pay_df[
    [
        "order_id",
        "order_status",
        "total_payment_value",
        "payment_count",
        "payment_installments",
        "primary_payment_type",
        "item_count",
        "total_item_price",
        "customer_state"
    ]
].head(10)

,order_id,order_status,total_payment_value,payment_count,payment_installments,primary_payment_type,item_count,total_item_price,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,38.71,3.0,1.0,credit_card,1.0,29.99,SP
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,141.46,1.0,1.0,boleto,1.0,118.70,BA
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,179.12,1.0,3.0,credit_card,1.0,159.90,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,72.20,1.0,1.0,credit_card,1.0,45.00,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,28.62,1.0,1.0,credit_card,1.0,19.90,SP
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,175.26,1.0,6.0,credit_card,1.0,147.90,PR
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,65.95,1.0,1.0,credit_card,1.0,49.90,RS
7,6514b8ad8028c9f2cc2374ded245783f,delivered,75.16,1.0,3.0,credit_card,1.0,59.99,RJ
8,76c6e866289321a7c93b82b54852dc33,delivered,35.95,1.0,1.0,boleto,1.0,19.90,RS
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,169.76,2.0,1.0,voucher,1.0,149.99,SP


## Feature Engineering

**We derive temporal, payment, order-value, and operational features
from the order-level dataset for revenue recovery modeling.**

In [27]:
# Make a working copy
model_df = revive_pay_df.copy()

# Convert timestamps to datetime
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    model_df[col] = pd.to_datetime(model_df[col], errors="coerce")

# Time features
model_df["purchase_year"] = model_df["order_purchase_timestamp"].dt.year
model_df["purchase_month"] = model_df["order_purchase_timestamp"].dt.month
model_df["purchase_day"] = model_df["order_purchase_timestamp"].dt.day
model_df["purchase_dayofweek"] = model_df["order_purchase_timestamp"].dt.dayofweek
model_df["purchase_hour"] = model_df["order_purchase_timestamp"].dt.hour

# Delivery-related features
model_df["approval_delay_hours"] = (
    model_df["order_approved_at"]
    - model_df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

model_df["delivery_time_days"] = (
    model_df["order_delivered_customer_date"]
    - model_df["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

model_df["estimated_delivery_gap_days"] = (
    model_df["order_estimated_delivery_date"]
    - model_df["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

print("Feature engineering completed.")
print("New shape:", model_df.shape)

Feature engineering completed.
New shape: (99441, 28)


In [28]:
model_df[
    [
        "order_status",
        "total_payment_value",
        "item_count",
        "payment_count",
        "purchase_month",
        "purchase_dayofweek",
        "approval_delay_hours",
        "delivery_time_days",
        "estimated_delivery_gap_days"
    ]
].head(10)

,order_status,total_payment_value,item_count,payment_count,purchase_month,purchase_dayofweek,approval_delay_hours,delivery_time_days,estimated_delivery_gap_days
0,delivered,38.71,1.0,3.0,10,0,0.178333,8.436574,15.544063
1,delivered,141.46,1.0,1.0,7,1,30.713889,13.782037,19.137766
2,delivered,179.12,1.0,1.0,8,2,0.276111,9.394213,26.639711
3,delivered,72.20,1.0,1.0,11,5,0.298056,13.208750,26.188819
4,delivered,28.62,1.0,1.0,2,1,1.030556,2.873877,12.112049
5,delivered,175.26,1.0,1.0,7,6,0.218889,16.542245,22.085359
6,invoiced,65.95,1.0,1.0,4,1,49.052500,NaN,27.484630
7,delivered,75.16,1.0,1.0,5,1,0.194722,9.989826,21.451042
8,delivered,35.95,1.0,1.0,1,0,32.360556,9.818762,41.229757
9,delivered,169.76,1.0,2.0,7,5,0.175000,18.221852,24.503449


## Defining Revenue Recovery Opportunities

The Olist dataset does not provide actual post-intervention recovery
outcomes. Therefore, Revive Pay uses historical unsuccessful order
outcomes as a proxy for revenue-at-risk opportunities.

Canceled and unavailable orders are initially treated as recovery
opportunities when an associated payment value exists.

In [29]:
# Define recovery opportunity
model_df["recovery_opportunity"] = (
    model_df["order_status"].isin(["canceled", "unavailable"])
    & model_df["total_payment_value"].fillna(0).gt(0)
).astype(int)

print(
    model_df["recovery_opportunity"]
    .value_counts()
)

recovery_opportunity
0    98210
1     1231
Name: count, dtype: int64


In [30]:
opportunity_summary = (
    model_df
    .groupby("recovery_opportunity")["total_payment_value"]
    .agg(
        order_count="count",
        total_value="sum",
        average_value="mean",
        median_value="median"
    )
)

opportunity_summary

,order_count,total_value,average_value,median_value
recovery_opportunity,,,,
0,98209,15739137.01,160.261656,105.28
1,1231,269735.11,219.118692,107.96


In [31]:
model_df["revenue_at_risk"] = np.where(
    model_df["recovery_opportunity"] == 1,
    model_df["total_payment_value"],
    0
)

In [32]:
model_df[
    [
        "order_status",
        "total_payment_value",
        "recovery_opportunity",
        "revenue_at_risk"
    ]
].head(20)

,order_status,total_payment_value,recovery_opportunity,revenue_at_risk
0,delivered,38.71,0,0.0
1,delivered,141.46,0,0.0
2,delivered,179.12,0,0.0
3,delivered,72.20,0,0.0
4,delivered,28.62,0,0.0
5,delivered,175.26,0,0.0
6,invoiced,65.95,0,0.0
7,delivered,75.16,0,0.0
8,delivered,35.95,0,0.0
9,delivered,169.76,0,0.0


In [33]:
print("Total revenue at risk:",
      round(model_df["revenue_at_risk"].sum(), 2))

print("Recovery opportunities:",
      model_df["recovery_opportunity"].sum())

Total revenue at risk: 269735.11
Recovery opportunities: 1231


## ML Dataset Preparation

We prepare leakage-free features for detecting historical recovery
opportunities and estimating the associated recovery value.